# Simulación de ingeniería de datos con datos de la NBA

Este cuaderno desarrolla una simulación completa de ingeniería de datos. El tema elegido es el rendimiento histórico de jugadores de la NBA por temporada. El flujo incluye extracción, limpieza, organización, movimiento entre bases de datos, transformación y presentación de resultados para un dashboard.

## 1. Tema y objetivo

**Tema:** estadísticas históricas de jugadores NBA.

**Objetivo:** construir un pipeline que convierta un CSV crudo en tablas analíticas útiles para responder preguntas como: quiénes son los jugadores con mayor producción, cómo cambian las métricas por temporada y qué equipos concentran más rendimiento.

## 2. Origen y obtención de datos

La fuente principal es un CSV público alojado en GitHub:

`https://raw.githubusercontent.com/illumitata/NBA/master/data/Seasons_Stats.csv`

El archivo contiene estadísticas por jugador y temporada. La extracción se realiza automáticamente con Python. Si no hay conexión, se usa una muestra mínima de respaldo para que el pipeline pueda ejecutarse.

In [ ]:
import io
import sqlite3
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = 'https://raw.githubusercontent.com/illumitata/NBA/master/data/Seasons_Stats.csv'
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

fallback_csv = '''Year,Player,Pos,Age,Tm,G,PER,PTS,TRB,AST,STL,BLK
2022,Nikola Jokic,C,26,DEN,74,32.8,2004,1019,584,109,63
2022,Giannis Antetokounmpo,PF,27,MIL,67,32.1,2002,778,388,72,91
2022,Luka Doncic,PG,22,DAL,65,25.1,1847,593,568,75,36
2021,Stephen Curry,PG,32,GSW,63,26.3,2015,345,363,77,8
2021,Joel Embiid,C,26,PHI,51,30.3,1451,539,145,50,69
2020,James Harden,SG,30,HOU,68,29.1,2335,446,512,125,60
2019,Paul George,SF,28,OKC,77,23.3,2159,628,318,170,34
2018,LeBron James,SF,33,CLE,82,28.6,2251,709,747,116,71
2017,Russell Westbrook,PG,28,OKC,81,30.6,2558,864,840,132,31
2016,Kevin Durant,SF,27,OKC,72,28.2,2029,589,361,69,85
'''

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        raw_bytes = response.read()
    source = 'GitHub'
    raw_df = pd.read_csv(io.BytesIO(raw_bytes))
except Exception as exc:
    source = f'muestra de respaldo: {exc.__class__.__name__}'
    raw_df = pd.read_csv(io.StringIO(fallback_csv))

print(f'Fuente usada: {source}')
print(f'Filas crudas: {len(raw_df):,}')
raw_df.head()

## 3. Carga en una base staging

La capa **staging** guarda el dataset tal como llega, con el menor número posible de cambios. Esto permite auditar el origen y repetir transformaciones sin volver a descargar.

In [ ]:
staging_db = DATA_DIR / 'nba_staging.sqlite'
with sqlite3.connect(staging_db) as conn:
    raw_df.to_sql('raw_seasons_stats', conn, if_exists='replace', index=False)

print(f'Datos crudos cargados en: {staging_db}')

## 4. Limpieza y organización

Reglas aplicadas:

- Seleccionar columnas relevantes para análisis.
- Normalizar nombres y tipos de datos.
- Eliminar registros sin año, jugador o equipo.
- Excluir filas agregadas `TOT` para evitar doble conteo por jugadores transferidos.
- Quitar duplicados por temporada, jugador y equipo.
- Rellenar métricas numéricas vacías con `0`.

In [ ]:
columns = ['Year', 'Player', 'Pos', 'Age', 'Tm', 'G', 'PER', 'PTS', 'TRB', 'AST', 'STL', 'BLK']
available = [c for c in columns if c in raw_df.columns]
clean_df = raw_df[available].copy()

for col in ['Year', 'Age', 'G', 'PER', 'PTS', 'TRB', 'AST', 'STL', 'BLK']:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')

clean_df['Player'] = clean_df['Player'].astype(str).str.replace('*', '', regex=False).str.strip()
clean_df['Pos'] = clean_df['Pos'].fillna('NA').astype(str).str.strip()
clean_df['Tm'] = clean_df['Tm'].fillna('NA').astype(str).str.strip()

clean_df = clean_df.dropna(subset=['Year', 'Player', 'Tm'])
clean_df = clean_df[clean_df['Tm'] != 'TOT']
clean_df = clean_df.drop_duplicates(subset=['Year', 'Player', 'Tm'])
clean_df[['Age', 'G', 'PER', 'PTS', 'TRB', 'AST', 'STL', 'BLK']] = clean_df[['Age', 'G', 'PER', 'PTS', 'TRB', 'AST', 'STL', 'BLK']].fillna(0)
clean_df['Year'] = clean_df['Year'].astype(int)

print(f'Filas limpias: {len(clean_df):,}')
clean_df.head()

## 5. Movimiento hacia una base analítica

Ahora movemos los datos desde staging hacia una base **analytics**. Esta segunda base contiene tablas listas para consumo por dashboards, reportes o modelos.

In [ ]:
analytics_db = DATA_DIR / 'nba_analytics.sqlite'
with sqlite3.connect(analytics_db) as conn:
    clean_df.to_sql('player_season_clean', conn, if_exists='replace', index=False)

print(f'Datos limpios cargados en: {analytics_db}')

## 6. Transformaciones analíticas

Creamos tablas agregadas para responder preguntas de análisis. Esta etapa cambia el formato de datos crudos a modelos útiles para visualización.

In [ ]:
team_season_summary = (
    clean_df.groupby(['Year', 'Tm'], as_index=False)
    .agg(players=('Player', 'nunique'), avg_pts=('PTS', 'mean'), avg_trb=('TRB', 'mean'), avg_ast=('AST', 'mean'), avg_per=('PER', 'mean'))
    .sort_values(['Year', 'avg_pts'], ascending=[False, False])
)

season_summary = (
    clean_df.groupby('Year', as_index=False)
    .agg(players=('Player', 'nunique'), teams=('Tm', 'nunique'), avg_pts=('PTS', 'mean'), avg_trb=('TRB', 'mean'), avg_ast=('AST', 'mean'), avg_per=('PER', 'mean'))
    .sort_values('Year')
)

top_players = clean_df.sort_values('PTS', ascending=False).head(20)

with sqlite3.connect(analytics_db) as conn:
    team_season_summary.to_sql('team_season_summary', conn, if_exists='replace', index=False)
    season_summary.to_sql('season_summary', conn, if_exists='replace', index=False)
    top_players.to_sql('top_players', conn, if_exists='replace', index=False)

season_summary.tail()

## 7. Resultados finales

Las tablas finales son:

- `player_season_clean`: hechos limpios por jugador y temporada.
- `team_season_summary`: agregados por equipo y temporada.
- `season_summary`: agregados por temporada.
- `top_players`: ranking de jugadores por puntos.

Estas tablas alimentan el dashboard entregado en `dashboard_nba.html`.

In [ ]:
print('Top 10 jugadores por puntos:')
top_players[['Year', 'Player', 'Tm', 'PTS', 'TRB', 'AST', 'PER']].head(10)

## 8. Reflexión final

El proceso permitió entender que un pipeline de ingeniería de datos necesita más que una descarga de archivos. Primero se identifica una fuente, luego se conserva una copia cruda, se aplican reglas de limpieza, se mueven datos entre capas y finalmente se transforman en tablas analíticas. El dashboard demuestra el valor de esa preparación porque permite filtrar, comparar y comunicar resultados de forma rápida.